In [3]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score     
from sklearn.metrics import roc_auc_score

In [4]:
import os
print(os.getcwd())

c:\Users\yhe\Desktop\PHD\Research\25spring_Agentic_AI_project\data


In [5]:
stroke_data = pd.read_csv("../data/stroke_dataset/stroke.csv")
stroke_data

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


Select numerical features

In [6]:
numeric_features = stroke_data.select_dtypes(include=['number']).columns.tolist()
numeric_features

['id',
 'age',
 'hypertension',
 'heart_disease',
 'avg_glucose_level',
 'bmi',
 'stroke']

In [7]:
categorical_features = [col for col in stroke_data.columns if col not in numeric_features]
categorical_features

['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

Remove the only one instances with gender to be "Other"

In [8]:
stroke_data = stroke_data[stroke_data.gender != 'Other'].copy()
stroke_data.groupby('gender')['gender'].count()
stroke_data.groupby('Residence_type')['Residence_type'].count()
stroke_data.groupby('smoking_status')['smoking_status'].count()

smoking_status
Unknown            1544
formerly smoked     884
never smoked       1892
smokes              789
Name: smoking_status, dtype: int64

Drop nan rows and delete useless id informaiton

In [9]:
stroke_data.dropna(inplace=True)
stroke_data.drop(columns=["id"],inplace=True)

In [10]:
stroke_data['ever_married'] = stroke_data['ever_married'].astype('category')
print(stroke_data['ever_married'].cat.categories)
print(enumerate(stroke_data['ever_married'].cat.categories))
print(dict(enumerate(stroke_data['ever_married'].cat.categories)))

Index(['No', 'Yes'], dtype='object')
{0: 'No', 1: 'Yes'}


Encoding categorical features

In [11]:
randomstate=123

def encode_features(df, categorical_features):
    binary_mappings = {}
    for feature in categorical_features:
        unique_values = df[feature].nunique()
        if unique_values == 2:
            # Binary encoding
            df[feature] = df[feature].astype('category') 
            mapping = dict(enumerate(df[feature].cat.categories)) 
            binary_mappings[feature] = {v: k for k, v in mapping.items()} 
            df[feature] = df[feature].map(binary_mappings[feature])
            df[feature] = df[feature].astype('int')
        else:
            # One-hot encoding
            one_hot = pd.get_dummies(df[feature], prefix=feature, dtype="int")
            df = df.drop(feature, axis=1)
            df = pd.concat([df, one_hot], axis=1)
    return df, binary_mappings

# Apply encoding
stroke_data , binary_mappings= encode_features(stroke_data, categorical_features)

In [12]:
#Move target to end
target_col = stroke_data.pop("stroke") # pop(), remove the target colum.
stroke_data.insert(len(stroke_data.columns), "stroke", target_col)

Check columns

In [13]:
stroke_data.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'Residence_type', 'avg_glucose_level', 'bmi', 'work_type_Govt_job',
       'work_type_Never_worked', 'work_type_Private',
       'work_type_Self-employed', 'work_type_children',
       'smoking_status_Unknown', 'smoking_status_formerly smoked',
       'smoking_status_never smoked', 'smoking_status_smokes', 'stroke'],
      dtype='object')

In [14]:
feature_order=['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'Residence_type', 'avg_glucose_level', 'bmi', 'work_type_Govt_job',
       'work_type_Never_worked', 'work_type_Private',
       'work_type_Self-employed', 'work_type_children',
       'smoking_status_Unknown', 'smoking_status_formerly smoked',
       'smoking_status_never smoked', 'smoking_status_smokes', 'stroke']

In [ ]:
#Create and save final train/test sets with target being the last column
stroke_data=stroke_data[feature_order]
#add dash between the string and lowercase all
stroke_data.columns = (
    stroke_data.columns
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.lower()
)

stroke_train,stroke_test = train_test_split(stroke_data, test_size=0.2 , random_state=randomstate)

stroke_train.to_parquet("stroke_dataset/train_cleaned.parquet")
stroke_test.to_parquet("stroke_dataset/test_cleaned.parquet")

x_train = stroke_train.drop('stroke', axis=1)
y_train= stroke_train['stroke']

x_test=stroke_test.drop('stroke', axis=1)
y_test=stroke_test['stroke']

model = RandomForestClassifier(n_estimators=100, random_state=randomstate)
# model=GaussianNB()

model.fit(x_train, y_train)

target_pred = model.predict(x_test)
target_prob=model.predict_proba(x_test)

accuracy = accuracy_score(y_test, target_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")
auc=roc_auc_score(y_test, model.predict_proba(x_test)[:, 1])
print(auc)

Accuracy: 95.72%
0.8213110403397028


In [16]:
with open('../data/stroke_dataset/RF.pkl', 'wb') as f:
    pickle.dump(model, f)

In [17]:
x_test.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'residence_type', 'avg_glucose_level', 'bmi', 'work_type_govt_job',
       'work_type_never_worked', 'work_type_private',
       'work_type_self_employed', 'work_type_children',
       'smoking_status_unknown', 'smoking_status_formerly_smoked',
       'smoking_status_never_smoked', 'smoking_status_smokes'],
      dtype='object')

In [18]:
feature_desc = [
    "The gender of the patient as a binary variable (0: female, 1: male)",
    "The age of the patient in years",
    "0 if the patient doesn't have hypertension, 1 if the patient has hypertension",
    "0 if the patient doesn't have any heart diseases, 1 if the patient has a heart disease",
    "If the patient has ever married (0: No, 1: Yes)",
    'The residence type of the patient (0: Rural; 1: Urban)',
    "average glucose level in blood",
    "body mass index",
    "One-hot variable for patient's work type -- Government job",
    "One-hot variable for patient's work type -- Never worked",
    "One-hot variable for patient's work type -- Private",
    "One-hot variable for patient's work type -- Self-employed",
    "One-hot variable for patient's work type -- Children (The patient is a child)",
    "One-hot variable for patient's smoking status -- Unkown",
    "One-hot variable for patient's smoking status -- Formerly smoked",
    "One-hot variable for patient's smoking status -- Never smoked",
    "One-hot variable for patient's smoking status -- Smokes"
]


feature_desc_df = pd.DataFrame({
    "feature_name": list(x_test.columns),
    "feature_average": x_train.mean().to_list() ,
    "feature_desc": feature_desc,
})
     

dataset_description="This dataset is used to predict whether a patient is likely to get stroke based on the input parameters like gender, age, various diseases, and smoking status. Each row in the data provides relavant information about the patient."
target_description="This dataset is used to predict whether a patient is likely to get stroke (1) or not (0)."
task_description="Predict whether a patient is likely to get stroke (1) or not (0)."

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('../data/stroke_dataset/dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [19]:
feature_desc_df

,feature_name,feature_average,feature_desc
0,gender,0.413907,The gender of the patient as a binary variable...
1,age,42.909373,The age of the patient in years
2,hypertension,0.092461,"0 if the patient doesn't have hypertension, 1 ..."
3,heart_disease,0.050688,0 if the patient doesn't have any heart diseas...
4,ever_married,0.653591,"If the patient has ever married (0: No, 1: Yes)"
5,residence_type,0.504840,The residence type of the patient (0: Rural; 1...
6,avg_glucose_level,105.012759,average glucose level in blood
7,bmi,28.901707,body mass index
8,work_type_govt_job,0.131431,One-hot variable for patient's work type -- Go...
9,work_type_never_worked,0.003566,One-hot variable for patient's work type -- Ne...
